In [1]:
import pyodbc
import pandas as pd
import numpy as np

print(pyodbc.drivers())

['SQL Server', 'SQL Server Native Client 11.0', 'B1CRHPROXY', 'ODBC Driver 13 for SQL Server', 'Microsoft Access Driver (*.mdb, *.accdb)', 'Microsoft Excel Driver (*.xls, *.xlsx, *.xlsm, *.xlsb)', 'Microsoft Access Text Driver (*.txt, *.csv)', 'Microsoft Access dBASE Driver (*.dbf, *.ndx, *.mdx)', 'ODBC Driver 18 for SQL Server', 'SQL Server Native Client RDA 11.0', 'ODBC Driver 17 for SQL Server']


In [2]:
from sqlalchemy import create_engine, text
from urllib.parse import quote_plus
from dotenv import load_dotenv
import os
import pyodbc

load_dotenv()

SERVER = os.getenv("DB_SERVER1")
DBNAME = os.getenv("DB_NAME1")
USER = os.getenv("DB_USER1")
PASS = os.getenv("DB_PASSWORD1")
ENCRYPT = os.getenv("DB_ENCRYPT", "yes")
TRUST = os.getenv("DB_TRUST_CERT", "yes")

SERVER = SERVER.replace("\\\\", "\\")  # normaliza


cnx = (
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={SERVER};"
    f"DATABASE={DBNAME};"
    f"UID={USER};"
    f"PWD={PASS};"
    f"Encrypt={ENCRYPT};TrustServerCertificate={TRUST};"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={quote_plus(cnx)}")

In [3]:
from sqlalchemy import text
try:
    with engine.connect() as conn:
        print("Ping:", conn.execute(text("SELECT 1")).scalar_one())
        print("Version:", conn.execute(text("SELECT @@VERSION")).scalar_one())
        print("✅ Conexión OK")
except Exception as e:
    print("❌ Error al conectar:", e)

Ping: 1
Version: Microsoft SQL Server 2017 (RTM) - 14.0.1000.169 (X64) 
	Aug 22 2017 17:04:49 
	Copyright (C) 2017 Microsoft Corporation
	Standard Edition (64-bit) on Windows Server 2019 Standard 10.0 <X64> (Build 17763: )

✅ Conexión OK


In [33]:
import pandas as pd
# 🔥 TEST 2: traer datos a pandas
query = "SELECT TOP 10 * FROM OITM"

df = pd.read_sql(query, engine)

df.head(5)

,ItemCode,ItemName,FrgnName,ItmsGrpCod,CstGrpCode,VatGourpSa,CodeBars,VATLiable,PrchseItem,SellItem,...,U_CI_15_Familia,U_CI_16_CaMeHP,U_CI_16_Ratio,U_CI_16_DiEjeSal,U_CI_14_DiExtRoMM,U_CI_14_NoCotCo,U_CI_14_AnchoMM,U_CI_14_SentR,U_CI_14_DiEjeIN,U_CI_14_TiRod
0,"# 111A-30"" X 1.5",NaN,NaN,100,-1,NaN,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
1,"# 111A-30"" X 1.5""",BANDA TRANSPORTADORA CON ESPUMA Y GUIA,160,106,-1,,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
2,"# 111A-54"" X 1.5",NaN,NaN,100,-1,NaN,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
3,"# 111A-54"" X 1.5""",BANDA TRANSPORTADORA CON ESPUMA Y GUIA,160,106,-1,,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None
4,#108GL,BANDA TRANSPORTADORA 2PLY POLYCR57 WHITE PU GL...,160,106,-1,,None,Y,Y,Y,...,None,None,None,None,None,None,None,None,None,None


In [34]:
ruta_UDF_sap="./Datos/campos_UDF_SAPB1.csv"
df.to_csv(ruta_UDF_sap)

In [28]:
query = """
WITH Precios AS (
    SELECT
        T0.ItemCode,

        MAX(CASE WHEN T0.PriceList = 14 THEN T0.Price END) AS Precio_B2B_Estandar,
        MAX(CASE WHEN T0.PriceList = 12 THEN T0.Price END) AS Precio_B2B_VIP,
        MAX(CASE WHEN T0.PriceList = 13 THEN T0.Price END) AS Precio_B2C,
        MAX(CASE WHEN T0.PriceList = 11 THEN T0.Price END) AS Precio_Promociones_Ecommerce

    FROM ITM1 T0
    WHERE
        T0.PriceList IN (11, 12, 13, 14)
    GROUP BY
        T0.ItemCode
),

VentasRaw AS (
    -- ============================================================
    -- FACTURAS DE CLIENTES
    -- Ventas históricas del artículo en todas las bodegas
    -- ============================================================
    SELECT
        T1.ItemCode,
        YEAR(T0.DocDate) AS Anio,
        MONTH(T0.DocDate) AS Mes,
        SUM(T1.Quantity) AS Cantidad_Vendida
    FROM OINV T0
    INNER JOIN INV1 T1
        ON T0.DocEntry = T1.DocEntry
    WHERE
        T0.CANCELED = 'N'
        AND T0.DocDate >= '2024-01-01'
    GROUP BY
        T1.ItemCode,
        YEAR(T0.DocDate),
        MONTH(T0.DocDate)

    UNION ALL

    -- ============================================================
    -- NOTAS DE CRÉDITO DE CLIENTES
    -- Se restan de las ventas
    -- ============================================================
    SELECT
        T1.ItemCode,
        YEAR(T0.DocDate) AS Anio,
        MONTH(T0.DocDate) AS Mes,
        SUM(T1.Quantity * -1) AS Cantidad_Vendida
    FROM ORIN T0
    INNER JOIN RIN1 T1
        ON T0.DocEntry = T1.DocEntry
    WHERE
        T0.CANCELED = 'N'
        AND T0.DocDate >= '2024-01-01'
    GROUP BY
        T1.ItemCode,
        YEAR(T0.DocDate),
        MONTH(T0.DocDate)
),

Ventas AS (
    SELECT
        ItemCode,

        -- ============================================================
        -- 2024
        -- ============================================================
        SUM(CASE WHEN Anio = 2024 AND Mes = 1  THEN Cantidad_Vendida ELSE 0 END) AS Enero_2024,
        SUM(CASE WHEN Anio = 2024 AND Mes = 2  THEN Cantidad_Vendida ELSE 0 END) AS Febrero_2024,
        SUM(CASE WHEN Anio = 2024 AND Mes = 3  THEN Cantidad_Vendida ELSE 0 END) AS Marzo_2024,
        SUM(CASE WHEN Anio = 2024 AND Mes = 4  THEN Cantidad_Vendida ELSE 0 END) AS Abril_2024,
        SUM(CASE WHEN Anio = 2024 AND Mes = 5  THEN Cantidad_Vendida ELSE 0 END) AS Mayo_2024,
        SUM(CASE WHEN Anio = 2024 AND Mes = 6  THEN Cantidad_Vendida ELSE 0 END) AS Junio_2024,
        SUM(CASE WHEN Anio = 2024 AND Mes = 7  THEN Cantidad_Vendida ELSE 0 END) AS Julio_2024,
        SUM(CASE WHEN Anio = 2024 AND Mes = 8  THEN Cantidad_Vendida ELSE 0 END) AS Agosto_2024,
        SUM(CASE WHEN Anio = 2024 AND Mes = 9  THEN Cantidad_Vendida ELSE 0 END) AS Septiembre_2024,
        SUM(CASE WHEN Anio = 2024 AND Mes = 10 THEN Cantidad_Vendida ELSE 0 END) AS Octubre_2024,
        SUM(CASE WHEN Anio = 2024 AND Mes = 11 THEN Cantidad_Vendida ELSE 0 END) AS Noviembre_2024,
        SUM(CASE WHEN Anio = 2024 AND Mes = 12 THEN Cantidad_Vendida ELSE 0 END) AS Diciembre_2024,
        SUM(CASE WHEN Anio = 2024 THEN Cantidad_Vendida ELSE 0 END) AS Total_2024,

        -- ============================================================
        -- 2025
        -- ============================================================
        SUM(CASE WHEN Anio = 2025 AND Mes = 1  THEN Cantidad_Vendida ELSE 0 END) AS Enero_2025,
        SUM(CASE WHEN Anio = 2025 AND Mes = 2  THEN Cantidad_Vendida ELSE 0 END) AS Febrero_2025,
        SUM(CASE WHEN Anio = 2025 AND Mes = 3  THEN Cantidad_Vendida ELSE 0 END) AS Marzo_2025,
        SUM(CASE WHEN Anio = 2025 AND Mes = 4  THEN Cantidad_Vendida ELSE 0 END) AS Abril_2025,
        SUM(CASE WHEN Anio = 2025 AND Mes = 5  THEN Cantidad_Vendida ELSE 0 END) AS Mayo_2025,
        SUM(CASE WHEN Anio = 2025 AND Mes = 6  THEN Cantidad_Vendida ELSE 0 END) AS Junio_2025,
        SUM(CASE WHEN Anio = 2025 AND Mes = 7  THEN Cantidad_Vendida ELSE 0 END) AS Julio_2025,
        SUM(CASE WHEN Anio = 2025 AND Mes = 8  THEN Cantidad_Vendida ELSE 0 END) AS Agosto_2025,
        SUM(CASE WHEN Anio = 2025 AND Mes = 9  THEN Cantidad_Vendida ELSE 0 END) AS Septiembre_2025,
        SUM(CASE WHEN Anio = 2025 AND Mes = 10 THEN Cantidad_Vendida ELSE 0 END) AS Octubre_2025,
        SUM(CASE WHEN Anio = 2025 AND Mes = 11 THEN Cantidad_Vendida ELSE 0 END) AS Noviembre_2025,
        SUM(CASE WHEN Anio = 2025 AND Mes = 12 THEN Cantidad_Vendida ELSE 0 END) AS Diciembre_2025,
        SUM(CASE WHEN Anio = 2025 THEN Cantidad_Vendida ELSE 0 END) AS Total_2025,

        -- ============================================================
        -- 2026
        -- ============================================================
        SUM(CASE WHEN Anio = 2026 AND Mes = 1  THEN Cantidad_Vendida ELSE 0 END) AS Enero_2026,
        SUM(CASE WHEN Anio = 2026 AND Mes = 2  THEN Cantidad_Vendida ELSE 0 END) AS Febrero_2026,
        SUM(CASE WHEN Anio = 2026 AND Mes = 3  THEN Cantidad_Vendida ELSE 0 END) AS Marzo_2026,
        SUM(CASE WHEN Anio = 2026 AND Mes = 4  THEN Cantidad_Vendida ELSE 0 END) AS Abril_2026,
        SUM(CASE WHEN Anio = 2026 AND Mes = 5  THEN Cantidad_Vendida ELSE 0 END) AS Mayo_2026,
        SUM(CASE WHEN Anio = 2026 AND Mes = 6  THEN Cantidad_Vendida ELSE 0 END) AS Junio_2026,
        SUM(CASE WHEN Anio = 2026 AND Mes = 7  THEN Cantidad_Vendida ELSE 0 END) AS Julio_2026,
        SUM(CASE WHEN Anio = 2026 AND Mes = 8  THEN Cantidad_Vendida ELSE 0 END) AS Agosto_2026,
        SUM(CASE WHEN Anio = 2026 AND Mes = 9  THEN Cantidad_Vendida ELSE 0 END) AS Septiembre_2026,
        SUM(CASE WHEN Anio = 2026 AND Mes = 10 THEN Cantidad_Vendida ELSE 0 END) AS Octubre_2026,
        SUM(CASE WHEN Anio = 2026 AND Mes = 11 THEN Cantidad_Vendida ELSE 0 END) AS Noviembre_2026,
        SUM(CASE WHEN Anio = 2026 AND Mes = 12 THEN Cantidad_Vendida ELSE 0 END) AS Diciembre_2026,
        SUM(CASE WHEN Anio = 2026 THEN Cantidad_Vendida ELSE 0 END) AS Total_2026,

        SUM(Cantidad_Vendida) AS Total_Periodo

    FROM VentasRaw
    GROUP BY
        ItemCode
)

SELECT TOP 50

    -- ============================================================
    -- IDENTIFICACIÓN
    -- ============================================================
    T0.ItemCode AS Codigo_Articulo,
    T0.ItemName AS Descripcion_Completa,
    T0.ItmsGrpCod AS Codigo_Grupo_Articulo,
    T1.ItmsGrpNam AS Grupo_Articulo,
    T0.InvntryUom AS Unidad_Medida_Inventario,
    T0.FirmCode AS Codigo_Fabricante,
    T2.FirmName AS Marca_Fabricante,

    -- ============================================================
    -- COMPRAS / PROVEEDOR
    -- ============================================================
    T0.CardCode AS Codigo_Proveedor_Principal,
    T3.CardName AS Proveedor_Principal,
    T0.LastPurDat AS Ultima_Fecha_Compra,
    T0.LastPurPrc AS Ultimo_Precio_Compra,

    -- ============================================================
    -- COSTOS
    -- ============================================================
    T0.AvgPrice AS Costo_Promedio_Articulo,
    T4.AvgPrice AS Costo_Promedio_Bodega,

    -- ============================================================
    -- INVENTARIO BODEGA 14
    -- ============================================================
    T4.WhsCode AS Codigo_Bodega,
    T4.OnHand AS Stock_En_Mano_Bodega,
    T4.IsCommited AS Stock_Comprometido_Bodega,
    T4.OnOrder AS Stock_En_Pedido_Transito_Bodega,
    T4.MinStock AS Stock_Minimo_Bodega,
    T4.MaxStock AS Stock_Maximo_Bodega,

    T4.OnHand - T4.IsCommited + T4.OnOrder AS Stock_Disponible_Teorico,

    -- ============================================================
    -- PRECIOS POR LISTA
    -- ============================================================
    P.Precio_B2B_Estandar,
    P.Precio_B2B_VIP,
    P.Precio_B2C,
    P.Precio_Promociones_Ecommerce,

    -- ============================================================
    -- VENTAS 2024
    -- ============================================================
    ISNULL(V.Enero_2024, 0) AS Enero_2024,
    ISNULL(V.Febrero_2024, 0) AS Febrero_2024,
    ISNULL(V.Marzo_2024, 0) AS Marzo_2024,
    ISNULL(V.Abril_2024, 0) AS Abril_2024,
    ISNULL(V.Mayo_2024, 0) AS Mayo_2024,
    ISNULL(V.Junio_2024, 0) AS Junio_2024,
    ISNULL(V.Julio_2024, 0) AS Julio_2024,
    ISNULL(V.Agosto_2024, 0) AS Agosto_2024,
    ISNULL(V.Septiembre_2024, 0) AS Septiembre_2024,
    ISNULL(V.Octubre_2024, 0) AS Octubre_2024,
    ISNULL(V.Noviembre_2024, 0) AS Noviembre_2024,
    ISNULL(V.Diciembre_2024, 0) AS Diciembre_2024,
    ISNULL(V.Total_2024, 0) AS Total_2024,

    -- ============================================================
    -- VENTAS 2025
    -- ============================================================
    ISNULL(V.Enero_2025, 0) AS Enero_2025,
    ISNULL(V.Febrero_2025, 0) AS Febrero_2025,
    ISNULL(V.Marzo_2025, 0) AS Marzo_2025,
    ISNULL(V.Abril_2025, 0) AS Abril_2025,
    ISNULL(V.Mayo_2025, 0) AS Mayo_2025,
    ISNULL(V.Junio_2025, 0) AS Junio_2025,
    ISNULL(V.Julio_2025, 0) AS Julio_2025,
    ISNULL(V.Agosto_2025, 0) AS Agosto_2025,
    ISNULL(V.Septiembre_2025, 0) AS Septiembre_2025,
    ISNULL(V.Octubre_2025, 0) AS Octubre_2025,
    ISNULL(V.Noviembre_2025, 0) AS Noviembre_2025,
    ISNULL(V.Diciembre_2025, 0) AS Diciembre_2025,
    ISNULL(V.Total_2025, 0) AS Total_2025,

    -- ============================================================
    -- VENTAS 2026
    -- ============================================================
    ISNULL(V.Enero_2026, 0) AS Enero_2026,
    ISNULL(V.Febrero_2026, 0) AS Febrero_2026,
    ISNULL(V.Marzo_2026, 0) AS Marzo_2026,
    ISNULL(V.Abril_2026, 0) AS Abril_2026,
    ISNULL(V.Mayo_2026, 0) AS Mayo_2026,
    ISNULL(V.Junio_2026, 0) AS Junio_2026,
    ISNULL(V.Julio_2026, 0) AS Julio_2026,
    ISNULL(V.Agosto_2026, 0) AS Agosto_2026,
    ISNULL(V.Septiembre_2026, 0) AS Septiembre_2026,
    ISNULL(V.Octubre_2026, 0) AS Octubre_2026,
    ISNULL(V.Noviembre_2026, 0) AS Noviembre_2026,
    ISNULL(V.Diciembre_2026, 0) AS Diciembre_2026,
    ISNULL(V.Total_2026, 0) AS Total_2026,

    ISNULL(V.Total_Periodo, 0) AS Total_Periodo

FROM OITM T0

LEFT JOIN OITB T1
    ON T0.ItmsGrpCod = T1.ItmsGrpCod

LEFT JOIN OMRC T2
    ON T0.FirmCode = T2.FirmCode

LEFT JOIN OCRD T3
    ON T0.CardCode = T3.CardCode

INNER JOIN OITW T4
    ON T0.ItemCode = T4.ItemCode
    AND T4.WhsCode = '14'

LEFT JOIN Precios P
    ON T0.ItemCode = P.ItemCode

INNER JOIN Ventas V
    ON T0.ItemCode = V.ItemCode

WHERE
    T0.SellItem = 'Y'
    AND T0.PrchseItem = 'Y'
    AND T0.InvntItem = 'Y'
    AND T0.validFor = 'Y'
    AND T0.frozenFor = 'N'
    AND T2.FirmName LIKE '%SIEMENS%'
    AND ISNULL(V.Total_Periodo, 0) > 0

ORDER BY
    ISNULL(V.Total_Periodo, 0) DESC;
"""

In [31]:
df = pd.read_sql(query, engine)

df.head(5)

,Codigo_Articulo,Descripcion_Completa,Codigo_Grupo_Articulo,Grupo_Articulo,Unidad_Medida_Inventario,Codigo_Fabricante,Marca_Fabricante,Codigo_Proveedor_Principal,Proveedor_Principal,Ultima_Fecha_Compra,...,Mayo_2026,Junio_2026,Julio_2026,Agosto_2026,Septiembre_2026,Octubre_2026,Noviembre_2026,Diciembre_2026,Total_2026,Total_Periodo
0,3UG45011AW30,rele de vigilancia analogico vigilancia de niv...,105,EQUI.AUTO.Y CONTROL,UNI,39,SIEMENS,EPN0113,SIEMENS S.A.,2025-12-20,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,32.0
1,6ES72324HD320XB0,"SIMATIC S7-1200, Salida analógica, SM 1232, 4 ...",105,EQUI.AUTO.Y CONTROL,UNI,39,SIEMENS,PEX1096,DIELLE - PLC CITY,2025-09-26,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.0
2,3UX00020AA02,ELECTRODO P/SUPERVISION NIVEL,105,EQUI.AUTO.Y CONTROL,UNI,39,SIEMENS,EPN0113,SIEMENS S.A.,2023-12-06,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0
3,3UG55141BR20,variante con ajuste analógico de relé de vigil...,105,EQUI.AUTO.Y CONTROL,UNI,39,SIEMENS,EPN0113,SIEMENS S.A.,2024-08-30,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.0
4,3UG46151CR20,¡ Producto a extinguir! El sucesor preferido e...,105,EQUI.AUTO.Y CONTROL,UNI,39,SIEMENS,EPN0113,SIEMENS S.A.,2026-04-09,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,3.0


In [30]:
ruta_query_sap="./Datos/Prueba_campos_maestros_SAP.csv"
df.to_csv(ruta_query_sap)